# 最後の1人：経験量 × モデル幅（CPU / CUDA）

幅64 / 128、各人2,048 → 8,192 → 32,768判断を比較する入口です。独立128人・6人レイド・ボス01・元の戦闘報酬。戦闘は既存の手書き方策です。

**作成者はColabを起動していません。** 利用者がセルを実行します。最初は環境構築 → 短いテスト → CPU/CUDA × 幅64/128の各1更新計測。長時間学習セルは既定で無効です。

GPU版を比較する場合は、先にColabの「ランタイムのタイプを変更」でGPUを選びます。CUDA版TorchはCPUでも使えるので、同じインストールで4条件を計測できます。


In [ ]:
# 1. 固定した実装を取得（学習は開始しない）
import subprocess
import sys
from pathlib import Path

CODE_REF = "01c101063fe7a1b77aa3031baea2f110363ad964"  # 固定コミット。mainへ置き換えない。
REPO = Path("/content/nmmo-research")
VENV = Path("/content/nmmo-scaling-env")
TORCH_BACKEND = "cu126"  # GPUを使わない環境では "cpu" も選べる

def call(args):
    return subprocess.run([str(a) for a in args], cwd=REPO, check=True)

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout",
                    "https://github.com/KOSEIHAMAYA2077/nmmo-research.git", str(REPO)], check=True)
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    # --no-checkout直後はHEADの追跡ファイルが未展開のため、indexの有無で判定する。
    if (REPO / ".git" / "index").exists():
        raise RuntimeError("作業フォルダに変更があります。別のREPOパスを指定してください。")
call(["git", "sparse-checkout", "set", "--no-cone", "/nmmo2/", "/colab/",
      "/runs/2026-11-16_last_picker_frozen/schedule.jsonl"])
call(["git", "fetch", "--depth", "1", "origin", CODE_REF])
call(["git", "checkout", "--detach", CODE_REF])
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == CODE_REF
print("Code:", CODE_REF)


In [ ]:
# 2. 別Python 3.10.16環境を用意。Colabカーネルはそのまま。
call([sys.executable, REPO / "colab/setup_last_picker.py", "--venv", VENV,
      "--torch-backend", TORCH_BACKEND])
PYTHON = VENV / "bin/python"
SCHEDULE = REPO / "runs/2026-11-16_last_picker_frozen/schedule.jsonl"
MODULE = "nmmo2.raid_nmmo.scaling_last_picker"


In [ ]:
# 3. 永続保存先。Driveの認証は利用者が行う。
from google.colab import drive
from datetime import datetime, timezone, timedelta

drive.mount("/content/drive")
DURABLE_ROOT = Path("/content/drive/MyDrive/NMMO/last_picker_scaling")
LOCAL_ROOT = Path("/content/nmmo-scaling-runs")
DURABLE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
def stamp():
    return datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d_%H%M%S")
print("Drive:", DURABLE_ROOT)


## 短い検証

CPUでは連続2更新と、1更新で保存して再開した場合を照合します。次の抽選、モデル、Adam、乱数、操作ログ、損失が一致するテストです。幅64／128の実モデルと合成戦闘で再開を照合し、研究用の学習系列に加算しません。ほかの補助テストには小さいモデルも使います。GPUがあればCUDAの収集・更新・評価・再開テストも動きます。


In [ ]:
# 4. 検算と回帰テスト。結果とハッシュをDriveにも残す。
call([PYTHON, REPO / "colab/verify_last_picker.py", "--out",
      DURABLE_ROOT / ("verification_" + stamp() + ".json")])


## 各1更新の計測

同じ試合抽選と同じ初期値を使い、CPU / CUDA × 幅64 / 128を計測します。GPUの計時には同期を入れます。CPUとCUDAのサンプリング乱数・軌跡のビット一致は保証しません。

`update_seconds` は抽選からログ圧縮までの実時間、`ppo_update_s` はPPO更新です。チェックポイント保存時間は別列。最初の1更新はキャッシュやCUDAの初期化を含み得ます。CUDAがなければ該当2条件を `skipped` と記録し、CPUの結果だけ残します。全評価・Driveコピーは時間外挿に含みません。


In [ ]:
# 5. 利用者が True にしたときだけ短い計測を行う
DO_PROFILE = False
PROFILE_UPDATES = 1
PROFILE_PATH = DURABLE_ROOT / ("profile_" + stamp() + ".json")
if DO_PROFILE:
    call([PYTHON, "-m", MODULE, "profile", "--schedule", SCHEDULE,
          "--updates", PROFILE_UPDATES, "--devices", "cpu", "cuda", "--out", PROFILE_PATH])
    import csv
    with PROFILE_PATH.with_suffix(".csv").open() as f:
        for row in csv.DictReader(f):
            print(row)
else:
    print("計測は未実行。DO_PROFILE=True に変更してこのセルを実行。")


## 学習を開始・再開する場合

ここからは長時間学習です。準備した段階では起動しません。計測後に利用者が選びます。

1系列ずつ、まずSTOP_UPDATE=256（各人2,048判断）。両幅の最初の保存点を比較してから1024、4096へ進みます。同じ幅・seed・device・Drive保存先で再実行すると最後の完了済み更新から続きます。停止点は追加更新数ではなく累積値です。CPUからCUDAへ切り替える場合は別系列にします。

16更新ごとに状態とログをDriveへ保存。突然切断された場合は未保存分を再実行します。古い中間点の再開状態のみ整理し、最新2点・初期点・指定保存点を保持します。試合ログは残ります。同じrunを複数ランタイムで同時実行しないでください。


In [ ]:
# 6. 長時間学習は既定で無効
DO_TRAIN = False
WIDTH = 64                  # 64 / 128
SEED = 41                   # 41 / 42 / 43
DEVICE = "cpu"              # "cpu" / "cuda"
STOP_UPDATE = 256            # 256 / 1024 / 4096
RUN_NAME = f"w{WIDTH}_s{SEED}_{DEVICE}"
if DO_TRAIN:
    call([PYTHON, "-m", MODULE, "train", "--schedule", SCHEDULE,
          "--width", WIDTH, "--seed", SEED, "--device", DEVICE,
          "--stop-update", STOP_UPDATE, "--checkpoint-every", 16,
          "--out", LOCAL_ROOT / RUN_NAME, "--durable", DURABLE_ROOT / RUN_NAME])
else:
    print("長時間学習は起動していません。", RUN_NAME, STOP_UPDATE)


## 保存点の評価・集計

保存点では通常／H↔D職表示反転 × 確率選択／最大確率選択を自動評価します。切断で評価だけ未完了なら、学習セルの再実行時に先に埋めます。明示的に評価だけ行う入口もあります。

主指標は同じ本人がAでH、BでDを確定した対。個人別12対の分布、中央値、1対以上／全対正答の人数も出します。未完了の保存点は集計しません。


In [ ]:
# 7. 評価だけを再開する場合（任意）
DO_EVALUATE = False
EVALUATE_UPDATE = 256
if DO_EVALUATE:
    call([PYTHON, "-m", MODULE, "evaluate", "--out", LOCAL_ROOT / RUN_NAME,
          "--durable", DURABLE_ROOT / RUN_NAME, "--update", EVALUATE_UPDATE])


In [ ]:
# 8. 同じバックエンドの、完了した保存点だけを比較
DO_REPORT = False
REPORT_DEVICE = "cpu"
if DO_REPORT:
    run_dirs = sorted(p for p in DURABLE_ROOT.glob(f"w*_s*_{REPORT_DEVICE}")
                      if (p / "settings.json").exists())
    if not run_dirs:
        raise RuntimeError("完了した系列がまだありません")
    report_path = DURABLE_ROOT / ("comparison_" + REPORT_DEVICE + "_" + stamp())
    call([PYTHON, "-m", MODULE, "report", *run_dirs, "--out", report_path])
    print((report_path / "RESULTS.md").read_text())


詳しい保存仕様・コマンドは [利用手順](https://github.com/KOSEIHAMAYA2077/nmmo-research/blob/01c101063fe7a1b77aa3031baea2f110363ad964/colab/LAST_PICKER_SCALING.md)、研究条件は [設計書](https://github.com/KOSEIHAMAYA2077/nmmo-research/blob/01c101063fe7a1b77aa3031baea2f110363ad964/nmmo2/raid_nmmo/COLAB_LAST_PICKER_SCALING.md) を参照。

Colabでのインストール、CPU/CUDA速度、GPUでの再開一致、学習成績は未確認です。利用者の実行で記録してください。
